# Choosing a polynomial degree with cross-validation

One day of temperature readings, polynomials of degree 1 through 9, and the question the lesson
asks: which degree? Here you fit the polynomials, draw the training and validation error curves,
write 5-fold cross-validation from scratch, and let it pick the degree. By the end you will have
seen the training error vote for the wrong answer and the held-out error vote for the right one.

**How to work through it.** Each part asks for one function or class. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

A weather station on a hill logged the air temperature at 60 random moments over one day. Time
is measured in days, so 0 is midnight and 0.5 is noon. The true pattern is one slow wave, coldest
before dawn and warmest in the late afternoon, and gusts and passing clouds add noise with a
standard deviation of about one degree. Nothing below uses the true curve except the final plot.
Run this cell first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)
n = 60
t = np.sort(rng.uniform(0, 1, n))                              # time of day, in days
true_curve = lambda t: 14 - 6 * np.cos(2 * np.pi * (t - 0.2))
temp = true_curve(t) + rng.normal(0, 1.0, n)                   # degrees Celsius
degrees = list(range(1, 10))


def mse(y_true, y_pred):
    return float(np.mean((y_true - y_pred) ** 2))


plt.scatter(t, temp, s=18)
plt.xlabel("time (days)")
plt.ylabel("temperature (C)")
plt.show()
print(t.shape, temp.shape)

## Part 1 (coding task): Fit a polynomial and predict

The model family is polynomials in $t$, and the degree is the dial that sets how flexible the
model is. `np.polyfit(x, y, degree)` finds the least squares coefficients and
`np.polyval(coef, x_new)` evaluates them, which is all the fitting this notebook needs.

Write `fit_predict(x_train, y_train, x_new, degree)`. It fits a polynomial of the given degree to
the training points and returns its predictions at `x_new`, one per entry.

In [ ]:
def fit_predict(x_train, y_train, x_new, degree):
    """Predictions at x_new from a least squares polynomial of this degree fit to the training points."""
    # TODO
    raise NotImplementedError

In [ ]:
xs = np.array([0.0, 1.0, 2.0, 3.0])
assert np.allclose(fit_predict(xs, 2 * xs + 1, np.array([4.0, 10.0]), 1), [9.0, 21.0]), "a line through points on y = 2x + 1 should predict 9 and 21"
assert np.allclose(fit_predict(xs, np.array([1.0, 3.0, 2.0, 6.0]), xs, 0), 3.0), "degree 0 is a constant, the mean of the targets"
assert np.allclose(fit_predict(xs, xs ** 2, np.array([5.0]), 2), [25.0]), "degree 2 recovers y = x^2 exactly"
pred = fit_predict(t, temp, t, 3)
assert pred.shape == t.shape, f"one prediction per input: expected shape {t.shape}, got {pred.shape}"
print("Part 1 passed")

## Part 2 (coding task): Hold some readings back

Training error cannot choose the degree, because it only falls as the degree rises. The choice
needs readings the fit never saw. Shuffle the row indices once, keep some as a validation set, and
fit on the rest. Shuffling matters: the readings are sorted by time, so cutting them in order
would train on the morning and validate on the evening.

Write `train_val_split(n, n_val, seed)`. It returns `(train_idx, val_idx)`, two integer arrays
that together cover `0` to `n - 1` exactly once, with `n_val` indices in the validation set. Use
`np.random.default_rng(seed).permutation(n)` for the shuffle.

In [ ]:
def train_val_split(n, n_val, seed=0):
    """(train_idx, val_idx): a shuffled split of range(n) with n_val rows held out."""
    # TODO
    raise NotImplementedError

In [ ]:
train_idx, val_idx = train_val_split(60, 15, seed=0)
assert len(val_idx) == 15 and len(train_idx) == 45, f"15 held out of 60 leaves 45, got {len(val_idx)} and {len(train_idx)}"
assert np.array_equal(np.sort(np.concatenate([train_idx, val_idx])), np.arange(60)), "the two sets should cover every row exactly once"
assert not np.array_equal(np.sort(val_idx), np.arange(15)), "the split should be shuffled, not the first 15 rows"
assert np.array_equal(val_idx, train_val_split(60, 15, seed=0)[1]), "the same seed should give the same split"
assert not np.array_equal(val_idx, train_val_split(60, 15, seed=1)[1]), "a different seed should give a different split"
print("Part 2 passed")

## Part 3 (coding task): Training and validation error against the degree

For each degree, fit on the training rows and measure the mean squared error twice: on the
training rows and on the validation rows. The training curve has to fall at every step, because a
degree-$d$ polynomial is a special case of a degree-$(d+1)$ one. The validation curve is the one
that can turn back up.

Write `error_curves(x, y, train_idx, val_idx, degrees)`. It returns `(train_mse, val_mse)`, two
arrays with one entry per degree. The check plots both curves.

In [ ]:
def error_curves(x, y, train_idx, val_idx, degrees):
    """(train_mse, val_mse) arrays, one entry per degree, from fits on the training rows only."""
    # TODO
    raise NotImplementedError

In [ ]:
train_mse, val_mse = error_curves(t, temp, train_idx, val_idx, degrees)
assert train_mse.shape == (9,) and val_mse.shape == (9,), "one entry per degree"
assert np.all(np.diff(train_mse) <= 1e-6), "training error should never rise with the degree"
assert abs(train_mse[0] - 9.007) < 1e-2, f"the degree-1 training error should be about 9.007 (fit on the 45 training rows only), got {train_mse[0]:.3f}"
assert abs(val_mse[2] - 0.635) < 1e-2, f"the degree-3 validation error should be about 0.635, got {val_mse[2]:.3f}"
best_val = degrees[int(np.argmin(val_mse))]
assert degrees[int(np.argmin(train_mse))] == 9, "training error picks degree 9, the most flexible"
assert best_val == 3, f"validation error should pick degree 3 on these readings, got {best_val}"
assert val_mse[-1] > 1.5 * val_mse.min(), "degree 9 should do clearly worse than the best degree on the held-out readings"

plt.plot(degrees, train_mse, "o-", label="training")
plt.plot(degrees, val_mse, "o-", label="validation")
plt.yscale("log")
plt.xlabel("degree")
plt.ylabel("mean squared error")
plt.legend()
plt.show()
print(f"training error picks degree 9, validation error picks degree {best_val}")
print("Part 3 passed")

## Part 4 (coding task): Cut the rows into k folds

One split wastes the held-out rows and depends on which rows landed there. $k$-fold
cross-validation shuffles the rows once, cuts them into $k$ near-equal folds, and uses each fold
as the validation set in turn while the other $k - 1$ train. Every row is scored exactly once.

Write `kfold_indices(n, k, seed)`. It returns a list of `k` pairs `(train_idx, val_idx)`. Shuffle
with `np.random.default_rng(seed).permutation(n)`, and cut with `np.array_split`, which splits an
array into `k` near-equal pieces.

In [ ]:
def kfold_indices(n, k, seed=0):
    """k pairs (train_idx, val_idx): each shuffled fold held out once, the other k - 1 training."""
    # TODO
    raise NotImplementedError

In [ ]:
folds = kfold_indices(60, 5, seed=0)
assert len(folds) == 5, f"5 folds should give 5 pairs, got {len(folds)}"
for train_f, val_f in folds:
    assert len(val_f) == 12 and len(train_f) == 48, "60 rows in 5 folds is 12 held out and 48 training each time"
    assert np.array_equal(np.sort(np.concatenate([train_f, val_f])), np.arange(60)), "each pair should cover every row exactly once"
all_val = np.sort(np.concatenate([val_f for _, val_f in folds]))
assert np.array_equal(all_val, np.arange(60)), "every row should be held out exactly once across the folds"
assert not np.array_equal(np.sort(folds[0][1]), np.arange(12)), "the folds should be shuffled, not cut in time order"
sizes = [len(val_f) for _, val_f in kfold_indices(61, 4, seed=0)]
assert sorted(sizes) == [15, 15, 15, 16], f"61 rows in 4 folds should be sizes 15, 15, 15 and 16, got {sizes}"
print("Part 4 passed")

## Part 5 (coding task): The cross-validation score of one degree

For one degree, fit on each fold's training rows, score on its validation rows, and collect the
$k$ scores. Their mean is the cross-validation estimate of the test error. Their spread says how
much to trust it: two degrees whose means differ by less than one fold's standard deviation are
not really distinguishable.

Write `cv_mse(x, y, degree, folds)`. It returns an array of the `k` held-out errors, one per fold,
in the order of `folds`.

In [ ]:
def cv_mse(x, y, degree, folds):
    """The k held-out mean squared errors of a polynomial of this degree, one per fold."""
    # TODO
    raise NotImplementedError

In [ ]:
scores = cv_mse(t, temp, 3, folds)
assert scores.shape == (5,), f"5 folds should give 5 scores, got {scores.shape}"
assert np.all(scores > 0), "every fold's error should be positive"
assert abs(scores.mean() - 0.937) < 1e-2, f"the mean CV error of degree 3 should be about 0.937, got {scores.mean():.3f}"
assert abs(cv_mse(t, temp, 1, folds).mean() - 8.510) < 1e-2, f"the mean CV error of degree 1 should be about 8.510, got {cv_mse(t, temp, 1, folds).mean():.3f}"

print(f"{'degree':>6} {'CV MSE':>8} {'fold std':>9}")
for d in degrees:
    s = cv_mse(t, temp, d, folds)
    print(f"{d:>6} {s.mean():>8.3f} {s.std():>9.3f}")
print("Part 5 passed")

## Part 6 (coding task): Pick the degree

Score every candidate degree on the same folds and keep the one with the lowest mean. That is the
whole algorithm behind `GridSearchCV`. Then refit the chosen degree on all 60 readings, since
cross-validation only chose the degree and the final model should use every row.

Write `pick_degree(x, y, degrees, folds)`. It returns `(best_degree, means)`, the chosen degree
and the array of mean CV errors, one per degree. The check draws the chosen polynomial through the
readings next to the true curve and the degree-9 fit that training error would have picked.

In [ ]:
def pick_degree(x, y, degrees, folds):
    """(best_degree, means): the degree with the lowest mean CV error, and all the means."""
    # TODO
    raise NotImplementedError

In [ ]:
best, means = pick_degree(t, temp, degrees, folds)
assert len(means) == 9, "one mean per degree"
assert best == degrees[int(np.argmin(means))], "the chosen degree should be the one with the lowest mean"
assert best == 5, f"cross-validation should pick degree 5 on these readings, got {best}"
assert means[-1] > means[best - 1], "degree 9 should score worse than the chosen degree"
assert means[0] > 5 * means[best - 1], "a straight line should score far worse than the chosen degree"

grid = np.linspace(0, 1, 300)
plt.scatter(t, temp, s=18, label="readings")
plt.plot(grid, true_curve(grid), "k--", label="true curve")
plt.plot(grid, fit_predict(t, temp, grid, best), label=f"degree {best}, chosen by CV")
plt.plot(grid, fit_predict(t, temp, grid, 9), alpha=0.6, label="degree 9, chosen by training error")
plt.xlabel("time (days)")
plt.ylabel("temperature (C)")
plt.legend()
plt.show()
print(f"cross-validation picks degree {best}")
print("Part 6 passed")

## Solution

### Part 1: Fit a polynomial and predict

In [ ]:
def fit_predict(x_train, y_train, x_new, degree):
    """Predictions at x_new from a least squares polynomial of this degree fit to the training points."""
    coef = np.polyfit(x_train, y_train, degree)
    return np.polyval(coef, x_new)

### Part 2: Hold some readings back

In [ ]:
def train_val_split(n, n_val, seed=0):
    """(train_idx, val_idx): a shuffled split of range(n) with n_val rows held out."""
    order = np.random.default_rng(seed).permutation(n)
    return order[n_val:], order[:n_val]

### Part 3: Training and validation error against the degree

In [ ]:
def error_curves(x, y, train_idx, val_idx, degrees):
    """(train_mse, val_mse) arrays, one entry per degree, from fits on the training rows only."""
    train_mse, val_mse = [], []
    for d in degrees:
        train_mse.append(mse(y[train_idx], fit_predict(x[train_idx], y[train_idx], x[train_idx], d)))
        val_mse.append(mse(y[val_idx], fit_predict(x[train_idx], y[train_idx], x[val_idx], d)))
    return np.array(train_mse), np.array(val_mse)

### Part 4: Cut the rows into k folds

In [ ]:
def kfold_indices(n, k, seed=0):
    """k pairs (train_idx, val_idx): each shuffled fold held out once, the other k - 1 training."""
    order = np.random.default_rng(seed).permutation(n)
    folds = np.array_split(order, k)
    pairs = []
    for i in range(k):
        val_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        pairs.append((train_idx, val_idx))
    return pairs

### Part 5: The cross-validation score of one degree

In [ ]:
def cv_mse(x, y, degree, folds):
    """The k held-out mean squared errors of a polynomial of this degree, one per fold."""
    scores = []
    for train_f, val_f in folds:
        pred = fit_predict(x[train_f], y[train_f], x[val_f], degree)
        scores.append(mse(y[val_f], pred))
    return np.array(scores)

### Part 6: Pick the degree

In [ ]:
def pick_degree(x, y, degrees, folds):
    """(best_degree, means): the degree with the lowest mean CV error, and all the means."""
    means = np.array([cv_mse(x, y, d, folds).mean() for d in degrees])
    return degrees[int(np.argmin(means))], means